# Kaggle GPU Training - Real-Time Multimodal Incident Intelligence System

This notebook trains the anomaly pipeline and exports model artifacts for reuse in the project backend.

## Dataset Choice

Primary dataset source is aligned with the Kaggle notebook input page: [odins0n/video-anomaly-detection/input](https://www.kaggle.com/code/odins0n/video-anomaly-detection/input).

This notebook auto-detects the best matching UCF-Crime dataset directory under `/kaggle/input`.

Why this one:
- Closely matches surveillance incident detection tasks.
- Includes public-safety classes such as RoadAccidents, Robbery, Fighting, Assault, etc.
- Manageable size for Kaggle sessions compared with full raw-video mirrors.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if gpu_available else 0
print('CUDA available:', gpu_available)
print('Visible GPU count:', gpu_count)
if gpu_available:
    for i in range(gpu_count):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
else:
    print('Training will run on CPU')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/saksham-1304/VIGIL-AI-Visual-Intelligence-Global-Incident-Learning'
REPO_DIR = Path('/kaggle/working/incident-intel-repo')

if not REPO_DIR.exists():
    if REPO_URL == 'YOUR_GITHUB_REPO_URL':
        raise ValueError('Set REPO_URL to your GitHub repository URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'ml/requirements.txt'], check=True)

In [ ]:
from pathlib import Path
import subprocess
import sys

# Kaggle datasets are mounted under /kaggle/input
INPUT_ROOT = Path('/kaggle/input/datasets/odins0n')
OUTPUT_DIR = Path('/kaggle/working/incident-intel-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

preferred_names = [
    'ucf-crime-dataset',
    'crimeucfdataset',
    'ucf-crime-full',
    'ucf-crimes',
    'crimeucf',
]

def has_train_test_layout(path: Path) -> bool:
    return (path / 'Train').exists() and (path / 'Test').exists()

def detect_dataset_dir() -> Path:
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(
            f'Kaggle input mount not found at {INPUT_ROOT}. Add dataset in notebook settings.'
        )

    # Case 1: dataset is mounted directly as /kaggle/input/Train and /kaggle/input/Test
    if has_train_test_layout(INPUT_ROOT):
        return INPUT_ROOT

    # Case 2: dataset is mounted as /kaggle/input/<dataset-slug>
    for name in preferred_names:
        candidate = INPUT_ROOT / name
        if candidate.exists() and candidate.is_dir():
            return candidate

    # Case 3: unknown slug with UCF/Crime tokens
    dynamic_candidates = sorted(
        [
            p for p in INPUT_ROOT.iterdir()
            if p.is_dir() and ('ucf' in p.name.lower() or 'crime' in p.name.lower())
        ],
        key=lambda p: p.name.lower(),
    )
    if dynamic_candidates:
        return dynamic_candidates[0]

    available = sorted([p.name for p in INPUT_ROOT.iterdir() if p.is_dir()])
    raise FileNotFoundError(
        'No UCF-Crime dataset folder found under /kaggle/input. '
        f'Available folders: {available}. '
        'Add dataset odins0n/ucf-crime-dataset in Notebook Settings > Add data.'
    )

DATASET_DIR = detect_dataset_dir()
print('Selected Kaggle dataset directory:', DATASET_DIR)

# Memory-friendly + checkpointed + multi-GPU aware training
cmd = [
    sys.executable, 'scripts/kaggle_train.py',
    '--input-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'auto',
    '--epochs', '40',
    '--latent-dim', '64',
    '--batch-size', '96',
    '--frame-step', '1',
    '--max-images', '300000',
    '--amp',
    '--multi-gpu',
    '--checkpoint-dir', str(OUTPUT_DIR / 'checkpoints' / 'autoencoder'),
    '--checkpoint-interval', '1',
    '--num-workers', '2',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

output_dir = Path('/kaggle/working/incident-intel-output')
print('Output files:')
for p in sorted(output_dir.glob('*')):
    print('-', p.name, f'({p.stat().st_size/1024/1024:.2f} MB)')

checkpoint_dir = output_dir / 'checkpoints' / 'autoencoder'
if checkpoint_dir.exists():
    ckpts = sorted(checkpoint_dir.glob('*.pt'))
    print(f'\nCheckpoint files: {len(ckpts)}')
    if ckpts:
        print('Latest checkpoint:', ckpts[-1].name)

summary_path = output_dir / 'training_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nBest model:', summary.get('best_model'))
    print('Runtime:', summary.get('runtime'))
    print('Training config:', summary.get('training'))

## After Training

1. Download `/kaggle/working/incident-intel-output/incident_intel_training_outputs.zip`.
2. Keep heavy binaries out of Git commits by default.
3. Publish artifacts using GitHub Releases or DVC remote storage.